# 05b — Death and discharge as competing events

## tl;dr

Estimate unadjusted admission-level cumulative incidence of in-hospital death and live discharge through day 28. Include age/sex/era cause-specific Cox models with patient-cluster inference. Default inputs and outputs are entirely synthetic.

## Context & Methods

Code recorded in-hospital death as event 1 and live discharge as event 2, keeping continuous elapsed follow-up. Use the two-cause Aalen-Johansen product integral with a common pre-event risk set for tied death, discharge and censor times. The input hospital episodes have both endpoints observed; no out-of-hospital mortality is added.

### Key assumptions
- CIF curves are unadjusted; adjusted cause-specific hazard ratios answer a different question.
- Resample patients globally with replacement, carrying every admission and exposure group with them. Use 1,000 bootstrap draws for clinical runs (200 for the default synthetic notebook).
- Intervals are pointwise 95% percentile intervals, not simultaneous bands.
- Export only the fixed daily grid 0–28, not individual event times. The current Cox death model remains a cause-specific model; this is no automatic replacement by Fine–Gray.
- Death-time fallbacks retain the original discharge-time proxy; exact death timing is uncertain for those admissions.


## Data

Run from the repository root or `notebooks/` after `uv sync --locked`. The existing bounded BigQuery loader is used only for explicit `current` mode. Pseudonymous keys stay in memory; never print source rows. Use `local` mode to inspect a previously verified aggregate run without another query.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from survival_sensitivity import run_supplement, load_local_bundle, plot_supplement
%matplotlib inline
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

In [ ]:
# Default: artificial data only. Never display a patient-level DataFrame.
DATA_MODE = "synthetic"  # "synthetic", "current", or "local"
LOCAL_RESULT = REPO_ROOT / "reports/local_supplements/sensitivity_run02/aggregate_results.json"
BOOTSTRAP_REPLICATES = 1000 if DATA_MODE == "current" else 200

if DATA_MODE == "local":
    bundle = load_local_bundle(LOCAL_RESULT, REPO_ROOT, "competing")
elif DATA_MODE in {"synthetic", "current"}:
    bundle = run_supplement("competing", repo=REPO_ROOT, clinical=DATA_MODE == "current",
                            bootstrap_replicates=BOOTSTRAP_REPLICATES)
else:
    raise ValueError("Unknown DATA_MODE")
result = bundle["results"]["competing"]
print({"data_origin": bundle["data_origin"], "supplement_version": bundle["supplement_version"],
       "main_analysis_updated": bundle["main_analysis_updated"]})

## Results

The data-origin label above controls interpretation. Synthetic outputs validate execution only.

### Fixed-time cumulative incidence

In [ ]:
rows = []
for group in result["groups"]:
    for r in group["curve"]:
        if r["day"] in [7, 14, 28]:
            rows.append({"RAAS orders": group["raas_order_group"], "Day": int(r["day"]),
                         "Death CIF": f"{100*r['death_cif']:.1f}% ({100*r['death_lower_95']:.1f}–{100*r['death_upper_95']:.1f})",
                         "Discharge CIF": f"{100*r['discharge_cif']:.1f}% ({100*r['discharge_lower_95']:.1f}–{100*r['discharge_upper_95']:.1f})"})
display(pd.DataFrame(rows))

### Adjusted cause-specific associations

In [ ]:
rows = []
for cause, model in result["cause_specific_cox"].items():
    e = model["estimates"]["raas_pre_icu"]; ci = e["patient_cluster"]
    rows.append({"Cause": cause, "Admissions": model["n"], "RAAS HR": e["hr"],
                 "Cluster 95% CI": f"{ci['lower_95']:.3f}–{ci['upper_95']:.3f}", "Cluster p": ci["p"]})
display(pd.DataFrame(rows).style.format({"RAAS HR": "{:.3f}", "Cluster p": "{:.4g}"}))

### Visual comparison

In [ ]:
ANALYSIS = "competing"
figure = plot_supplement(bundle, ANALYSIS)
display(figure)
plt.close(figure)

## Takeaways

Read death and discharge curves together. A cause-specific HR describes the instantaneous rate while still alive and hospitalized; the CIF describes the observed probability of a hospital endpoint by a specified time. The unadjusted CIF difference does not establish a treatment effect. Published Kaplan-Meier curves must not be read as competing-risk-adjusted absolute death probabilities.

See [the supplement guide](../docs/SURVIVAL_SENSITIVITY_SUPPLEMENTS.md) for the execution commands, data handling, reference checks and limitations. Tracked notebook outputs remain stripped; keep executed clinical copies under ignored `reports/local_supplements/`.